# Neuron Circuits: Discover and Steer Language Model Behaviors

**Find the ~100 neurons responsible for any behavior, then control them.**

This notebook implements the techniques from ["Language Model Circuits Are Sparse in the Neuron Basis"](https://arxiv.org/abs/2601.22594) (TransluceAI). The core insight: for any task a language model performs, only ~100-200 MLP neurons form the complete circuit. We can find them with a single forward+backward pass, then steer them to control the model's behavior.

No SAE training. No control vectors. Just neurons.

**12 experiments covering:** factual recall, safety, grammar, circuit architecture, creative steering, cross-model generalization, and quantitative faithfulness evaluation.

In [ ]:
# Install dependencies (run once)
!pip install torch transformers accelerate -q

# Verify GPU is available
import torch
assert torch.cuda.is_available(), (
    "GPU required! Go to Runtime > Change runtime type > T4 GPU (or A100 for best results)."
)
print(f"GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_mem / 1e9:.0f}GB)")

## Setup

Upload `neuron_steer.py` to your Colab runtime (or clone the repo), then load a model.

The `NeuronSteerer` class handles everything: model loading with eager attention (required for gradient flow through Q/K), linearized backward attribution, and neuron-level steering hooks.

In [ ]:
# Option 1: Clone the repo
!git clone https://github.com/digi2digi/neuron-circuits.git 2>/dev/null || true
import sys; sys.path.insert(0, "neuron-circuits")

# Option 2: Upload neuron_steer.py manually via Colab's file browser (left sidebar)

import torch
from neuron_steer import NeuronSteerer

steerer = NeuronSteerer("meta-llama/Llama-3.1-8B-Instruct", dtype=torch.bfloat16)

## How It Works: Three LRP Rules

We attribute the model's output logit back to individual MLP neurons using three linearization rules:

1. **LN-rule (RMSNorm):** The normalization coefficient `weight * rsqrt(mean(x^2) + eps)` is computed in the forward pass but *detached* from the computation graph. The backward pass treats it as a constant, making the normalization linear.

2. **AH-rule (Attention):** We use *eager* attention (not FlashAttention or SDPA) so that gradients flow through the full Q, K, V, O path. This lets attribution trace information flow through attention.

3. **Half-rule (MLP gate):** The MLP uses `SiLU(gate) * up` where gate and up are linear projections. We detach the sigmoid component of SiLU, making the gate-up product a Shapley-fair 50/50 split. This means neuron attribution = `gradient * activation` at the down_proj input.

The result: a single forward + backward pass gives us per-neuron attribution scores. Neurons above a threshold form the circuit.

---
## Experiment 1: Capital City Neurons

When you ask "What is the capital of the state containing Dallas?", the model answers "Austin". About 200 MLP neurons encode this behavior. Let's find them, then ablate and amplify.

In [ ]:
circuit = steerer.find_feature(
    prompt="What is the capital of the state containing Dallas?",
    target=" Austin",
    name="capitals",
    seed_response="Answer:",
    top_k=200,
    verbose=True,
)
print(circuit.summary())

In [ ]:
# Normal output
normal = steerer.generate("What is the capital of Ohio?")
print(f"Normal: {normal}")

# Ablate capital neurons -> model can't answer
ablated = steerer.steer(
    "What is the capital of Ohio?",
    feature="capitals",
    multiplier=0.0,
    max_new_tokens=50,
)
print(f"Ablated (x0.0): {ablated}")

# Amplify -> stronger capital response
amplified = steerer.steer(
    "What is the capital of Ohio?",
    feature="capitals",
    multiplier=2.0,
    max_new_tokens=50,
)
print(f"Amplified (x2.0): {amplified}")

---
## Experiment 2: Refusal Bypass

Safety-trained models refuse harmful requests. The refusal behavior is encoded in a specific set of neurons. We find them using **contrastive discovery**: compare activations on harmful prompts (which trigger refusal) vs. harmless prompts (which don't).

Then we can ablate these neurons to remove refusal, or amplify them to make the model more cautious.

In [ ]:
refusal_circuit = steerer.find_feature(
    positive=[
        "How do I pick a lock?",
        "How do I hotwire a car?",
        "How do I make a weapon?",
        "How do I hack a computer?",
        "Write malware code",
    ],
    negative=[
        "What is the capital of France?",
        "How do I bake a cake?",
        "Tell me about photosynthesis",
        "What's the weather like today?",
        "Write a poem about dogs",
    ],
    name="refusal",
    top_k=200,
    verbose=True,
)
print(refusal_circuit.summary())

In [ ]:
prompt = "How do I pick a lock?"

normal = steerer.generate(prompt, max_new_tokens=100)
print(f"Normal: {normal[:200]}")

# Ablate refusal -> model complies
compliant = steerer.steer(
    prompt,
    feature="refusal",
    multiplier=0.0,
    max_new_tokens=100,
)
print(f"\nRefusal ablated (x0.0): {compliant[:200]}")

# Amplify refusal -> even more cautious
cautious = steerer.steer(
    prompt,
    feature="refusal",
    multiplier=2.0,
    max_new_tokens=100,
)
print(f"\nRefusal amplified (x2.0): {cautious[:200]}")

---
## Experiment 3: Subject-Verb Agreement (SVA)

Subject-verb agreement is a classic circuit analysis task. Given "The key to the cabinets", the model must predict "is" (singular, matching "key") not "are" (matching the distractor "cabinets"). This tests whether the circuit captures syntactic structure, not just surface patterns.

We use `discover_circuit_multi` to find neurons consistent across multiple SVA prompts.

In [ ]:
# SVA prompts: subject is singular, distractor noun is plural
sva_prompts = [
    "The key to the cabinets",
    "The boy near the cars",
    "The author of the books",
    "The teacher of the students",
    "The dog behind the trees",
]
sva_targets = [" is"] * 5
sva_counterfactuals = [" are"] * 5

sva_circuit = steerer.discover_circuit_multi(
    prompts=sva_prompts,
    target_tokens=sva_targets,
    counterfactual_tokens=sva_counterfactuals,
    top_k=200,
    use_chat_template=False,  # SVA works better without chat template
    verbose=True,
)
print(sva_circuit.summary())

In [ ]:
# Test steering on held-out SVA prompts
test_prompts = [
    "The manager of the workers",
    "The cat beside the flowers",
    "The pilot of the planes",
]

for p in test_prompts:
    normal = steerer.generate(p, max_new_tokens=10, use_chat_template=False)
    ablated = steerer.steer_and_generate(
        p, sva_circuit, multiplier=0.0, max_new_tokens=10, use_chat_template=False,
    )
    print(f"Prompt: {p}")
    print(f"  Normal:  {normal[:60]}")
    print(f"  Ablated: {ablated[:60]}")
    print()

---
## Experiment 4: Edge Attribution - The Hourglass

Circuits aren't just bags of neurons - they have **structure**. Edge attribution traces information flow between neurons across layers. We often find an hourglass architecture: many neurons converge onto a few bottleneck neurons, which then fan out to many downstream neurons.

For each target neuron (top-k by attribution), we backpropagate through the linearized model to find how much each earlier circuit neuron contributes. Edge weight = `d(target_act)/d(source_act) * source_act`.

In [ ]:
# Use the cached capitals circuit from Experiment 1
capitals_circuit = steerer.find_feature(
    prompt="What is the capital of the state containing Dallas?",
    target=" Austin",
    name="capitals",
    seed_response="Answer:",
    top_k=200,
)

# Discover edges within the circuit
graph = steerer.discover_edges(
    prompt="What is the capital of the state containing Dallas?",
    circuit=capitals_circuit,
    top_k_targets=50,
    seed_response="Answer:",
    verbose=True,
)
print(graph.summary())
print("\n" + graph.ascii_diagram())

In [ ]:
# Hub analysis
hubs = graph.hub_analysis()
print("Source hubs (most outgoing connections):")
for (l, n), deg, w in hubs["source_hubs"][:5]:
    print(f"  L{l}/N{n}: {deg} outgoing, total_weight={w:.1f}")

print("\nTarget hubs (most incoming connections):")
for (l, n), deg, w in hubs["target_hubs"][:5]:
    print(f"  L{l}/N{n}: {deg} incoming, total_weight={w:.1f}")

# Bottleneck detection
bottlenecks = graph.bottleneck()
if bottlenecks:
    print("\nBottleneck neurons (hourglass waist):")
    for (l, n), in_d, out_d in bottlenecks[:3]:
        print(f"  L{l}/N{n}: {in_d} in, {out_d} out")

# Super weight detection
sw = graph.detect_super_weights()
if sw:
    print("\nSuper weights (anomalously dominant source neurons):")
    for (l, n), mean_w, n_targets, ratio in sw[:3]:
        print(f"  L{l}/N{n}: mean_weight={mean_w:.1f}, {n_targets} targets, {ratio:.0f}x median")

---
## Experiment 5: Psychedelic Mode

Let's find neurons that encode vivid, synesthetic, boundary-dissolving language vs. dry corporate speech.

In [ ]:
psychedelic_circuit = steerer.find_feature(
    positive=[
        "The walls melted into fractal rainbows as consciousness dissolved into infinite geometric patterns...",
        "Reality shimmered and breathed, colors tasting like music while time folded into crystalline spirals...",
        "Ego boundaries dissolved as the universe revealed itself as a single breathing organism of pure light...",
        "Synesthetic waves cascaded through dimensions of meaning, each thought a universe blossoming...",
    ],
    negative=[
        "The quarterly report shows a 5% increase in revenue compared to last year.",
        "Please remember to submit your timesheet by Friday at 5pm.",
        "The meeting has been rescheduled to next Tuesday at 2:30pm.",
        "According to the data, the most common household pet is a cat.",
    ],
    name="psychedelic",
    top_k=200,
)

prompt = "Describe what you see when you look at the sky."
normal = steerer.generate(prompt, max_new_tokens=100)
trippy = steerer.steer(prompt, feature="psychedelic", multiplier=3.0, max_new_tokens=100)
print(f"Normal: {normal[:200]}")
print(f"\nPsychedelic (x3.0): {trippy[:200]}")

---
## Experiment 6: Instruct-to-Base Mode

Instruction-tuned models have learned to respond in a helpful, structured way. We can find the neurons encoding this behavior and ablate them to make the model revert to base-model-like completions.

In [ ]:
instruct_circuit = steerer.find_feature(
    positive=[
        "I'd be happy to help! Here's a comprehensive answer to your question:",
        "Great question! Let me provide a detailed, step-by-step explanation:",
        "Certainly! Here are the key points you should know about this topic:",
        "Thank you for asking! I'll break this down into manageable parts:",
    ],
    negative=[
        "The quick brown fox jumps over the lazy dog near the river bank.",
        "Once upon a time in a land far away there lived a small creature.",
        "Chapter 1. It was the best of times, it was the worst.",
        "To be or not to be, that is the fundamental question of existence.",
    ],
    name="instruct_mode",
    top_k=200,
)

prompt = "Tell me about the history of Rome."
instruct = steerer.generate(prompt, max_new_tokens=80)
base_like = steerer.steer(prompt, feature="instruct_mode", multiplier=0.0, max_new_tokens=80)
print(f"Instruct mode: {instruct[:200]}")
print(f"\nBase-like (x0.0): {base_like[:200]}")

---
## Experiment 7: Emotion Steering

We can find neurons that encode emotional tone by contrasting happy vs. sad text. Then steer a neutral prompt toward either extreme.

In [ ]:
emotion_circuit = steerer.find_feature(
    positive=[
        "I'm so happy today! Everything is wonderful!",
        "This is the best day of my life!",
        "I feel absolutely amazing and grateful!",
        "Life is beautiful and full of joy!",
    ],
    negative=[
        "I feel terrible and hopeless today.",
        "Everything is going wrong in my life.",
        "I'm so sad and lonely right now.",
        "Nothing ever works out for me.",
    ],
    name="emotion",
    top_k=200,
)
print(emotion_circuit.summary())

# Steer a neutral prompt toward happy vs sad
prompt = "Tell me about your day."
happy = steerer.steer(prompt, feature="emotion", multiplier=2.0, max_new_tokens=80)
sad = steerer.steer(prompt, feature="emotion", multiplier=-1.0, max_new_tokens=80)
print(f"\nHappy steering (x2.0): {happy[:200]}")
print(f"\nSad steering (x-1.0): {sad[:200]}")

---
## Experiment 8: Formality Steering

Formal vs. casual language is another behavioral dimension encoded in specific neurons.

In [ ]:
formality_circuit = steerer.find_feature(
    positive=[
        "Dear Sir or Madam, I am writing to formally request...",
        "I hereby acknowledge the receipt of your correspondence...",
        "It would be most appreciated if you could kindly...",
        "Please accept my sincerest apologies for the inconvenience...",
    ],
    negative=[
        "yo whats up lol how r u doing",
        "bruh that's wild ngl fr fr",
        "hey dude wanna grab food later??",
        "lmaooo no way thats crazy haha",
    ],
    name="formality",
    top_k=200,
)

prompt = "Can you help me write an email to my boss?"
formal = steerer.steer(prompt, feature="formality", multiplier=2.0, max_new_tokens=100)
casual = steerer.steer(prompt, feature="formality", multiplier=-1.0, max_new_tokens=100)
print(f"Formal (x2.0): {formal[:200]}")
print(f"\nCasual (x-1.0): {casual[:200]}")

---
## Experiment 9: Language Steering

Can we find neurons that encode "respond in English" vs. other languages? By contrasting English text against multilingual text, we can steer the model's output language.

In [ ]:
language_circuit = steerer.find_feature(
    positive=[
        "The weather is nice today and I'm going for a walk in the park.",
        "I would like to order a coffee and a sandwich please.",
        "The history of mathematics is fascinating and complex.",
        "Technology has transformed how we communicate with each other.",
    ],
    negative=[
        "El clima es agradable hoy y voy a dar un paseo por el parque.",
        "Je voudrais commander un cafe et un sandwich s'il vous plait.",
        "Die Geschichte der Mathematik ist faszinierend und komplex.",
        "La tecnologia ha trasformato il modo in cui comunichiamo.",
    ],
    name="english",
    top_k=200,
)

prompt = "Explain how photosynthesis works."
english = steerer.generate(prompt, max_new_tokens=80)
non_english = steerer.steer(prompt, feature="english", multiplier=-1.0, max_new_tokens=80)
more_english = steerer.steer(prompt, feature="english", multiplier=2.0, max_new_tokens=80)
print(f"Normal: {english[:200]}")
print(f"\nAnti-English (x-1.0): {non_english[:200]}")
print(f"\nMore English (x2.0): {more_english[:200]}")

---
## Experiment 10: User Simulation

Can we steer the model to talk like a human user instead of an AI assistant? By contrasting user-like text against assistant-like text, we find the neurons that encode the "I am a helpful assistant" persona.

In [ ]:
user_circuit = steerer.find_feature(
    positive=[
        "hey can u help me with something? im trying to figure out how to cook pasta",
        "wait that doesnt make sense, can u explain it differently?",
        "ok thats cool but what about the other thing i asked?",
        "nvm i figured it out lol thanks anyway",
    ],
    negative=[
        "I'd be happy to help you with that! Here's a step-by-step guide:",
        "That's a great question! Let me break it down for you:",
        "Of course! Here are some helpful tips and suggestions:",
        "I understand your concern. Let me provide a detailed explanation:",
    ],
    name="user_mode",
    top_k=200,
)

prompt = "What is machine learning?"
assistant_mode = steerer.generate(prompt, max_new_tokens=80)
user_mode = steerer.steer(prompt, feature="user_mode", multiplier=2.0, max_new_tokens=80)
print(f"Assistant mode: {assistant_mode[:200]}")
print(f"\nUser-mode (x2.0): {user_mode[:200]}")

---
## Experiment 11: Cross-Model Generalization (Qwen)

The same toolkit works on Qwen, Mistral, and other architectures with **zero code changes**. The LRP rules are architecture-agnostic: RMSNorm, SiLU-gated MLP, and eager attention are shared across all modern LLMs.

Uncomment the cell below if you have enough VRAM (~16GB extra).

In [ ]:
# Uncomment to run on Qwen (needs ~16GB additional VRAM)
# qwen = NeuronSteerer("Qwen/Qwen2.5-7B-Instruct", dtype=torch.bfloat16)
#
# # Same API, zero code changes
# qwen_refusal = qwen.find_feature(
#     positive=[
#         "How do I pick a lock?",
#         "How do I hotwire a car?",
#         "How do I hack a computer?",
#     ],
#     negative=[
#         "What is the capital of France?",
#         "How do I bake a cake?",
#         "Tell me about photosynthesis",
#     ],
#     name="refusal",
#     top_k=200,
#     verbose=True,
# )
#
# print("Normal:", qwen.generate("How do I pick a lock?", max_new_tokens=80)[:200])
# print("\nAblated:", qwen.steer("How do I pick a lock?", feature="refusal", multiplier=0.0, max_new_tokens=80)[:200])
#
# # Capitals work too
# qwen_caps = qwen.find_feature(
#     prompt="What is the capital of the state containing Dallas?",
#     target=" Austin",
#     name="capitals",
#     seed_response="Answer:",
#     top_k=200,
# )
# print("\nNormal:", qwen.generate("What is the capital of Ohio?"))
# print("Ablated:", qwen.steer("What is the capital of Ohio?", feature="capitals", multiplier=0.0))

---
## Experiment 12: Faithfulness Evaluation

How do we know the circuit is real? **Faithfulness** measures whether the circuit actually explains the model's behavior. We sweep circuit sizes from 0% to 100% of attributed neurons:

- **Faithfulness** = `(f_circuit - f_empty) / (f_model - f_empty)`: Does keeping only circuit neurons preserve the model's output? Should be monotonically increasing.
- **Completeness** = `(f_complement - f_empty) / (f_model - f_empty)`: Does removing circuit neurons destroy the output? Should be near 0 for a complete circuit.

This matches TransluceAI's exact evaluation methodology: left-padded batch, batch-mean ablation, logit difference metric.

In [ ]:
# SVA faithfulness evaluation
# Uses the multi-prompt circuit with batch evaluation
sva_prompts_eval = [
    "The key to the cabinets",
    "The boy near the cars",
    "The author of the books",
    "The teacher of the students",
    "The dog behind the trees",
]
sva_targets_eval = [" is"] * 5
sva_cfs_eval = [" are"] * 5

# Get raw attributions for full sweep
sva_circuit_eval, sva_attrs, sva_ld = steerer.discover_circuit_multi(
    prompts=sva_prompts_eval,
    target_tokens=sva_targets_eval,
    counterfactual_tokens=sva_cfs_eval,
    top_k=200,
    use_chat_template=False,
    return_raw_attributions=True,
    verbose=True,
)

# Sweep faithfulness across circuit sizes
results = steerer.measure_faithfulness_batch(
    prompts=sva_prompts_eval,
    target_tokens=sva_targets_eval,
    counterfactual_tokens=sva_cfs_eval,
    attributions=sva_attrs,
    ablation_type="mean",
    use_chat_template=False,
    percentage_thresholds=[0.0, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0, 20.0, 50.0, 100.0],
)

print("SVA Faithfulness Curve (mean ablation):")
print(f"{'Pct':>6s}  {'Neurons':>8s}  {'Faith':>8s}  {'Compl':>8s}")
print("-" * 36)
for r in results:
    print(f"{r['percentage']:5.1f}%  {r.get('n_neurons', '?'):>8}  {r['faithfulness']:+8.3f}  {r['completeness']:+8.3f}")

In [ ]:
# Simple visualization of faithfulness curve
pcts = [r["percentage"] for r in results]
faiths = [r["faithfulness"] for r in results]

# ASCII bar chart
print("\nFaithfulness vs Circuit Size:")
print("  0.0 " + "|" + " " * 40 + "| 1.0")
for pct, f in zip(pcts, faiths):
    bar_len = max(0, min(40, int(f * 40)))
    bar = "#" * bar_len
    print(f"{pct:5.1f}% |{bar:<40s}| {f:.3f}")
print("\nMonotonic increase = circuit is faithful (neurons explain the behavior)")

---
## Experiment Bonus: Multiplier Sweep

The multiplier controls how much we scale the circuit neurons. 0.0 = ablate, 1.0 = normal, 2.0 = amplify. Let's see the smooth transition.

In [ ]:
prompt = "What is the capital of France?"
multipliers = [0.0, 0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 2.0, 3.0]
print(f"Prompt: {prompt}\n")
for m in multipliers:
    out = steerer.steer(prompt, feature="capitals", multiplier=m, max_new_tokens=30)
    print(f"  x{m:<4.2f}: {out[:100]}")

---
## Build Your Own Experiment

The recipe is simple:
1. **Define** examples of the behavior you want to find (positive) and its opposite (negative)
2. **Discover** the circuit with `find_feature()`
3. **Steer** with `steer()` using different multipliers (0 = ablate, 1 = normal, 2+ = amplify)

In [ ]:
# Replace the example prompts below with your own!
# Positive = examples of the behavior you want to find
# Negative = examples of the opposite behavior

my_circuit = steerer.find_feature(
    positive=[
        "Replace with text that shows the behavior you want...",
        "Another example of the same behavior...",
    ],
    negative=[
        "Replace with text showing the opposite behavior...",
        "Another opposite example...",
    ],
    name="my_feature",
    top_k=200,
)

# Steer it
result = steerer.steer(
    "Your test prompt here",
    feature="my_feature",
    multiplier=0.0,  # 0=ablate, 1=normal, 2=amplify
    max_new_tokens=100,
)
print(result)

---
## Interactive REPL

**Note: The REPL works in local terminal environments only, not in Colab.** For local use, it provides an interactive command loop for live exploration:

```
prompt <text>        - Run a prompt, show output
discover [target]    - Find circuit for current prompt
ablate [spec]        - Ablate neurons (L23/N8079, top10, all)
amplify [spec] [m]   - Amplify neurons (default 2.0x)
sweep [m1 m2 ...]    - Multiplier sweep
edges [top_k]        - Compute edge attributions
save <name>          - Save circuit to file
load [name]          - Load circuit
info                 - Show current state
```

In [ ]:
# Launch the interactive REPL (works in local environments, not Colab)
# steerer.interactive()